In [1]:
#@title Environment-Aware Installation
import os
import sys
from IPython import get_ipython

# ── Detect environment ────────────────────────────────────────────────
ENV_NAME = None
BASE_DATA_PATH = None
BASE_OUTPUT_PATH = None
DATA_MOUNT = None
KAGGLE_WHEEL_DIR = None

os.environ["TORCH_CUDA_ARCH_LIST"] = "12.0"
!export CMAKE_CUDA_ARCHITECTURES="120"
try:
    if "google.colab" in str(get_ipython()):
        ENV_NAME = "colab"
        BASE_DATA_PATH = "/content/"
        BASE_OUTPUT_PATH = "/content/"
        DATA_MOUNT = "/content/ToolFormer/data/generated/v1.0_k5"
        print("Environment: Google Colab")
        print("Will use uv pip install (internet available)")
    elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        ENV_NAME = "kaggle"
        BASE_DATA_PATH = "/kaggle/input/"
        BASE_OUTPUT_PATH = "/kaggle/working/"
        DATA_MOUNT = "/kaggle/input/datasets/dzung271828/toolformer-data/generated/v1.0_k5"
        KAGGLE_WHEEL_DIR = "/kaggle/input/datasets/dzung271828/telco-wheels/telco-wheels/telco-wheels"
        print("Environment: Kaggle")
        print("Will use pip --no-index --find-links (offline mode)")
        # Prevent HF from trying to download
        os.environ["HF_DATASETS_OFFLINE"] = "1"
        os.environ["TRANSFORMERS_OFFLINE"] = "1"
        os.environ["HF_HUB_OFFLINE"] = "1"
    else:
        ENV_NAME = "local"
        BASE_DATA_PATH = "./data/"
        BASE_OUTPUT_PATH = "./output/"
        DATA_MOUNT = "data/generated/v1.0_k5"
        print("Environment: Local")
except NameError:
    ENV_NAME = "local"
    BASE_DATA_PATH = "./data/"
    BASE_OUTPUT_PATH = "./output/"
    DATA_MOUNT = "data/generated/v1.0_k5"
    print("Non-interactive session. Using local paths.")

os.makedirs(BASE_OUTPUT_PATH, exist_ok=True)
print(f"Environment: {ENV_NAME}")
print(f"Base data path: {BASE_DATA_PATH}")
print(f"Base output path: {BASE_OUTPUT_PATH}")
print(f"Data mount: {DATA_MOUNT}")
if KAGGLE_WHEEL_DIR:
    print(f"Kaggle wheel dir: {KAGGLE_WHEEL_DIR}")
    
# ── Detect GPU type ──────────────────────────────────────────────────
import subprocess as _sp
IS_T4_GPU = False
try:
    _gpu_name = (
        _sp.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            text=True,
        )
        .strip()
        .split("\n")[0]
    )
    IS_T4_GPU = "Tesla T4" in _gpu_name
    print(f"GPU type: {_gpu_name} → {'T4 (internet install)' if IS_T4_GPU else 'non-T4 (offline install)'}")
except Exception:
    print("GPU detection: nvidia-smi unavailable → defaulting to offline install")
print(f"IS_T4_GPU: {IS_T4_GPU}")

Environment: Kaggle
Will use pip --no-index --find-links (offline mode)
Environment: kaggle
Base data path: /kaggle/input/
Base output path: /kaggle/working/
Data mount: /kaggle/input/datasets/dzung271828/toolformer-data/generated/v1.0_k5
Kaggle wheel dir: /kaggle/input/datasets/dzung271828/telco-wheels/telco-wheels/telco-wheels
GPU type: NVIDIA RTX PRO 6000 Blackwell Server Edition → non-T4 (offline install)
IS_T4_GPU: False


In [2]:
# ── Install packages ────────────────────────────────────────────────
os.environ["UNSLOTH_VLLM_STANDBY"] = "0"  # Disable vLLM standby mode for training SFT

if ENV_NAME == "colab":
    print("Installing packages for colab env...")
    # Colab: full internet available, use uv pip
    !pip install --upgrade -qqq uv
    try:
        import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except:
        _numpy = "numpy"
        _pil = "pillow"
    try:
        import subprocess
        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except:
        is_t4 = False
    _vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
    !uv pip install transformers==4.56.2
    !uv pip install --no-deps trl==0.22.2
elif ENV_NAME == "kaggle":
    print("Installing packages for kaggle env...")
    # Kaggle: no internet, use pre-loaded wheels and datasets
    import subprocess

    subprocess.run(
        "pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages "
        "unsloth trl peft transformers datasets accelerate bitsandbytes vllm",
        shell=True,
        check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True,
        check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True,
        check=True,
    )
    for _wd in ["/kaggle/input/datasets/llkh0a/rtx-wheels/wheels"]:
        if os.path.isdir(_wd):
            subprocess.run(
                [
                    "pip",
                    "install",
                    "-q",
                    "--no-index",
                    "--find-links",
                    _wd,
                    "protobuf==6.33.5",
                    "sentencepiece",
                    "safetensors",
                    "huggingface_hub",
                    "vllm",
                ],
                check=False,
            )
    subprocess.run("rm -rf /kaggle/tmp/*", shell=True, check=True)
else:
    print("Installing packages for local env...")
    # Local: regular pip
    !pip install unsloth vllm
    !pip install transformers==4.56.2
    !pip install trl==0.22.2


# Kaggle: install pre-downloaded torch + flash-attn + einops (Blackwell wheels)
if ENV_NAME == "kaggle":
    !pip install --no-index --find-links=/kaggle/input/datasets/nctuan/nvidia-offline-packages-nemotron/ /kaggle/input/datasets/nctuan/nvidia-offline-packages-nemotron/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

print("Core stack installation complete.")

Installing packages for kaggle env...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
google-adk 1.25.1 requires opentelemetry-api<1.40.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.25.1 requires opentelemetry-sdk<1.40.0,>=1.36.0, but you have opentelem

Looking in links: /kaggle/input/datasets/nctuan/nvidia-offline-packages-nemotron/
Processing /kaggle/input/datasets/nctuan/nvidia-offline-packages-nemotron/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Core stack installation complete.


In [3]:
#@title Check installed library versions
import sys
import importlib

_LIBS = {
    "unsloth": None,
    "unsloth_zoo": None,
    "torch": None,
    "transformers": None,
    "trl": None,
    "peft": None,
    "vllm": None,
    "datasets": None,
    "accelerate": None,
    "bitsandbytes": None,
    "flash_attn": None,
}

print(f"{'Library':<20} {'Version':<20}")
print("-" * 40)
for lib_name in _LIBS:
    try:
        mod = importlib.import_module(lib_name)
        ver = getattr(mod, "__version__", "no __version__")
        print(f"{lib_name:<20} {ver:<20}")
    except ImportError:
        print(f"{lib_name:<20} {'not installed':<20}")

Library              Version             
----------------------------------------
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-08-21 17:55:11.303593: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787334911.485876      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787334911.541525      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787334911.997417      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787334911.997434      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787334911.997435      65 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth              2026.3.17           
unsloth_zoo          2026.3.6            
torch                2.10.0+cu128        
transformers         4.57.6              
trl                  0.24.0              
peft                 0.18.1              
vllm                 0.18.0              
datasets             4.3.0               
accelerate           1.12.0              
bitsandbytes         0.49.2              
flash_attn           2.8.3               


In [4]:
import os
import sys
from IPython import get_ipython

def load_secret(key_name: str) -> str | None:
    """Load a secret from environment-specific secret stores.

    Args:
        key_name: Name of the secret key to load.

    Returns:
        The secret value if found, otherwise None.
    """
    env = ENV_NAME
    secret_value = None
    print(f"Attempting to load secret '{key_name}' from '{env}' environment...")
    try:
        if env == "colab":
            from google.colab import userdata

            secret_value = userdata.get(key_name)
        elif env == "kaggle":
            from kaggle_secrets import UserSecretsClient

            user_secrets = UserSecretsClient()
            secret_value = user_secrets.get_secret(key_name)
        else:
            secret_value = os.getenv(key_name)
        if not secret_value:
            print(f"Secret '{key_name}' not found in the {env} environment.")
            return None
        print(f"Successfully loaded secret '{key_name}'.")
        return secret_value
    except Exception as e:
        print(f"An error occurred while loading secret '{key_name}': {e}")
        return None


def print_system_info():
    """Print Python version, PyTorch/CUDA info, GPU count and nvidia-smi output."""
    print("\n🔧 System Information")
    print(f"Python version: {sys.version.split()[0]}")
    try:
        import torch

        print(f"PyTorch version: {torch.__version__}")
        if torch.cuda.is_available():
            print(f"CUDA version: {torch.version.cuda}")
            print(f"GPU count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("CUDA not available")
    except ImportError:
        print("PyTorch not installed")
    finally:
        !nvidia-smi

is_kaggle = ENV_NAME == "kaggle"
is_colab = ENV_NAME == "colab"
is_local = ENV_NAME == "local"
print_system_info()

if not is_kaggle:
    os.environ["WANDB_API_KEY"] = wandb_key = load_secret("WANDB_API_KEY")
    os.environ["HF_TOKEN"] = HF_TOKEN = load_secret("HF_TOKEN")
    GITHUB_TOKEN = load_secret("GITHUB_TOKEN")


🔧 System Information
Python version: 3.12.12
PyTorch version: 2.10.0+cu128
CUDA version: 12.8
GPU count: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition
Fri Aug 21 17:56:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             86W /  600W |     681MiB /  97887MiB |  

In [5]:
!find /usr -name "libcuda.so*" 2>/dev/null

/usr/local/nvidia/lib64/libcuda.so.580.159.04
/usr/local/nvidia/lib64/libcuda.so.1
/usr/local/nvidia/lib64/libcuda.so
/usr/local/cuda-12.8/compat/libcuda.so.1
/usr/local/cuda-12.8/compat/libcuda.so
/usr/local/cuda-12.8/compat/libcuda.so.570.124.06


In [6]:
import os

# Define the directory where libcuda.so lives
nvidia_lib_dir = "/usr/local/nvidia/lib64"

# Export for the compiler linker (fixes -lcuda error)
os.environ["LIBRARY_PATH"] = nvidia_lib_dir + ":" + os.environ.get("LIBRARY_PATH", "")

# Export for the runtime dynamic linker
os.environ["LD_LIBRARY_PATH"] = nvidia_lib_dir + ":" + os.environ.get("LD_LIBRARY_PATH", "")

print("Environment paths for libcuda successfully configured!")
print(os.environ["LIBRARY_PATH"])
print(os.environ["LD_LIBRARY_PATH"])
!rm -rf ~/.cache/torch_extensions/

Environment paths for libcuda successfully configured!
/usr/local/nvidia/lib64:/usr/local/cuda/lib64/stubs
/usr/local/nvidia/lib64:/usr/local/lib/python3.12/dist-packages/cv2/../../lib64:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/cuda/lib64


### Unsloth
#
Goal: To convert `Qwen3-4B-Base` into a reasoning model via GRPO by using OpenR1's Math dataset.
#
We first pre fine-tune the model to make GRPO skip trying to match formatting - this speeds GRPO up.

In [7]:

# === Ablation / experiment configuration (CLI) ===
import argparse
import json
import pathlib
import sys


def parse_args(argv=None):
    p = argparse.ArgumentParser(
        description="Qwen3-4B GRPO math-reasoning fine-tuning (ablation-ready, WandB-tracked).",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    # --- Model / data paths (offline: point to /kaggle/input copies) ---
    p.add_argument(
        "--model-path",
        type=str,
        default="/kaggle/input/models/qwen-lm/qwen-3/transformers/1.7b-base/1",
        help="HF model id or local path (offline: /kaggle/input/...)",
    )
    p.add_argument(
        "--grpo-data-path",
        type=str,
        default="/kaggle/input/datasets/alejopaullier/openr1-math-220k",
        help="GRPO training dataset (offline: /kaggle/input/...)",
    )
    p.add_argument(
        "--sft-data-path",
        type=str,
        default="/kaggle/input/datasets/ga5534/openmathreasoning-cot-kaggle",
        help="Pre-SFT formatting dataset (offline: /kaggle/input/...)",
    )
    p.add_argument(
        "--sample-n",
        type=int,
        default=0,
        help="Subset to first N GRPO samples (0 = use all)",
    )
    p.add_argument(
        "--max-seq-length",
        type=int,
        default=16384,
        help="Max sequence length (context window)",
    )
    p.add_argument(
        "--sft-epochs", 
        type=int, 
        default=1, 
        help="Pre-SFT epochs")
    p.add_argument(
        "--no-sft",
        default=True,
        action="store_true",
        help="Skip pre-SFT formatting (GRPO-from-base comparison arm)",
    )

    # --- GRPO ablation knobs (each is an experimental axis) ---
    p.add_argument(
        "--beta",
        type=float,
        default=0.000,
        help="KL-divergence penalty coefficient (0 disables KL & its logging). "
        "Note: trl only logs `kl` when beta != 0.",
    )
    p.add_argument(
        "--num-generations",
        type=int,
        default=8,
        choices=[2, 4, 8, 16],
        help="Group size: completions sampled per prompt (num_generations)",
    )
    p.add_argument(
        "--importance-sampling-level",
        type=str,
        default="token",
        choices=["token", "sequence"],
        help="Advantage credit assignment granularity (token-level vs sequence-level)",
    )
    p.add_argument(
        "--loss-type",
        type=str,
        default="dapo",
        choices=["grpo", "bnpo", "dr_grpo", "dapo", "cispo"],
        help="GRPO loss variant (normalization differences)",
    )
    p.add_argument(
        "--temperature", type=float, default=1.0, help="Sampling temperature"
    )
    p.add_argument("--lr", type=float, default=1e-5, help="Learning rate")
    p.add_argument(
        "--optim",
        type=str,
        default="adamw_torch",
        choices=["adamw_torch", "adamw_8bit"],
        help="Optimizer (adamw default: bitsandbytes has a shape bug on Blackwell)",
    )
    p.add_argument("--max-steps", type=int, default=500, help="Training steps")
    p.add_argument("--seed", type=int, default=3407, help="Global seed")

    # --- Run identity / tracking ---
    p.add_argument(
        "--run-name",
        type=str,
        default=None,
        help="Run id (auto-generated from ablation axes if omitted)",
    )
    p.add_argument(
        "--wandb",
        dest="wandb",
        action="store_true",
        default=False,
        help="Enable WandB logging",
    )
    p.add_argument(
        "--no-wandb", dest="wandb", action="store_false", help="Disable WandB logging"
    )
    p.add_argument(
        "--wandb-project",
        type=str,
        default="grpo-math-ablation",
        help="WandB project (env WANDB_PROJECT overrides)",
    )
    p.add_argument(
        "--wandb-group",
        type=str,
        default="default",
        help="WandB run group (use ablation axis, e.g. beta, num_generations)",
    )
    p.add_argument(
        "--offline-mode",
        action="store_true",
        default=True,
        help="Force WandB offline (no network; sync later with `wandb sync`)",
    )
    return p.parse_args(args=["--offline-mode"])


args = parse_args()


RUN_NAME = args.run_name or "_".join(
    [
        f"beta{args.beta}",
        f"ng{args.num_generations}",
        f"loss_{args.loss_type}",
        f"is_{args.importance_sampling_level}",
        f"temp{args.temperature}",
        f"lr{args.lr}",
        f"opt{args.optim}",
    ]
    + (["noSFT"] if args.no_sft else [])
    + ([f"n{args.sample_n}"] if args.sample_n else [])
)
output_dir = pathlib.Path("outputs") / RUN_NAME
output_dir.mkdir(parents=True, exist_ok=True)

# --- WandB env (API key read from environment automatically) ---
if args.wandb:
    os.environ.setdefault("WANDB_API_KEY", os.environ.get("WANDB_API_KEY", ""))
    os.environ.setdefault("WANDB_PROJECT", args.wandb_project)
    os.environ.setdefault("WANDB_RUN_GROUP", args.wandb_group)
    os.environ.setdefault("WANDB_NAME", RUN_NAME)
    if args.offline_mode:
        os.environ["WANDB_MODE"] = "offline"
        print(
            "[wandb] Offline mode enabled (WANDB_MODE=offline) — sync later with `wandb sync`"
        )
    if not os.environ.get("WANDB_API_KEY"):
        print(
            "[wandb] WANDB_API_KEY not found in environment — falling back to offline mode "
            "(run dir saved under ./wandb; sync with `wandb sync` when online)"
        )
        os.environ["WANDB_MODE"] = "offline"

import os
import json
import pathlib
from typing import Tuple
import torch

# Giữ lại các biến toàn cục ENV_NAME, args, RUN_NAME, output_dir từ code cũ của bạn

def save_config_and_hyperparams(output_dir: pathlib.Path, args, tokenizer=None, extra: dict = None):
    """[Yêu cầu 4] Lưu config và hyperparam ngay khi bắt đầu train để tracking"""
    output_dir = pathlib.Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Save full args -> config.json
    config_path = output_dir / "config.json"
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(vars(args), f, indent=2, ensure_ascii=False, default=str)

    # 2. Save hyperparams quan trọng -> hyperparams.json
    hyperparams = {
        "model_path": args.model_path,
        "max_seq_length": args.max_seq_length,
        "lora_rank": 32, # lấy từ config của bạn
        "sft_epochs": args.sft_epochs,
        "learning_rate": args.lr,
        "beta": args.beta,
        "num_generations": args.num_generations,
        "loss_type": args.loss_type,
        "importance_sampling_level": args.importance_sampling_level,
        "temperature": args.temperature,
        "optim": args.optim,
        "max_steps": args.max_steps,
        "seed": args.seed,
        "run_name": RUN_NAME,
    }
    if extra:
        hyperparams.update(extra)

    hparams_path = output_dir / "hyperparams.json"
    with open(hparams_path, "w", encoding="utf-8") as f:
        json.dump(hyperparams, f, indent=2, ensure_ascii=False, default=str)

    print(f"[tracking] Saved config to {config_path}")
    print(f"[tracking] Saved hyperparams to {hparams_path}")

    # Nếu dùng wandb thì update luôn
    if args.wandb:
        try:
            import wandb
            wandb.config.update(hyperparams, allow_val_change=True)
        except:
            pass
    return hyperparams

def load_model(
    model_path: str,
    max_seq_length: int,
    lora_rank: int,
    seed: int,
    mode: str = "train", # [Yêu cầu 1] "train" hoặc "eval"
    stage: str = "sft", # "sft" | "grpo" | "eval"
) -> Tuple:
    """
    [Yêu cầu 1,2,3] 1 hàm duy nhất load model
        - mode = "train" | "eval"
        - stage = "sft" | "grpo"
    Logic:
        - train SFT -> standby OFF, fast_inference OFF
        - train GRPO -> standby OFF, fast_inference ON
        - eval -> standby OFF, fast_inference ON
    """
    assert mode in ("train", "eval"), "mode phải là 'train' hoặc 'eval'"
    assert stage in ("sft", "grpo", "eval"), "stage phải là 'sft' | 'grpo' | 'eval'"

    # Xác định trạng thái
    is_sft_train = (mode == "train" and stage == "sft")
    is_grpo_train = (mode == "train" and stage == "grpo")

    if is_sft_train:
        # [Yêu cầu 2] khi train SFT thì tắt standby và tắt fast_inference
        os.environ["UNSLOTH_VLLM_STANDBY"] = "0"
        fast_inference_flag = False
        print("[load_model] MODE: TRAIN-SFT -> UNSLOTH_VLLM_STANDBY=0, fast_inference=False")
    elif is_grpo_train:
        # [Yêu cầu 3] khi train GRPO thì bật standby và bật fast_inference
        os.environ["UNSLOTH_VLLM_STANDBY"] = "0"
        fast_inference_flag = True
        print("[load_model] MODE: TRAIN-GRPO -> UNSLOTH_VLLM_STANDBY=1, fast_inference=True")
    else: # eval
        os.environ["UNSLOTH_VLLM_STANDBY"] = "0"
        fast_inference_flag = True
        print(f"[load_model] MODE: {mode.upper()}-{stage.upper()} -> UNSLOTH_VLLM_STANDBY=0, fast_inference=True")

    # Import SAU khi set env - rất quan trọng với Unsloth
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=max_seq_length,
        load_in_4bit=False,
        fast_inference=fast_inference_flag,
        max_lora_rank=lora_rank,
        gpu_memory_utilization=0.6,
        attn_implementation="flash_attention_2",
        dtype=torch.bfloat16
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_rank,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_alpha=lora_rank * 2,
        use_gradient_checkpointing="unsloth",
        random_state=seed,
    )
    return model, tokenizer

### GRPO chat template
Since we're using a base model, we should set a chat template. You can make your own chat template as well!
1. DeepSeek uses `<think>` and `</think>`, but this is **not** necessary - you can customize it however you like!
2. A `system_prompt` is recommended to at least guide the model's responses.

In [8]:

reasoning_start = "<start_working_out>"  # Acts as think-open tag
reasoning_end = "<end_working_out>"  # Acts as think-close tag
solution_start = "<SOLUTION>"
solution_end = "</SOLUTION>"

system_prompt = f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>'

We create a simple chat template below. Notice `add_generation_prompt` includes prepending `<start_working_out>` to guide the model to start its reasoning process.

In [9]:
model, tokenizer = load_model(
        model_path=args.model_path,
        max_seq_length=args.max_seq_length,
        lora_rank=32,
        seed=args.seed,
        mode="train",
        stage="sft" # -> tự tắt standby + fast_inference
    )
chat_template = (
    "{% if messages[0]['role'] == 'system' %}"
    "{{ messages[0]['content'] + eos_token }}"
    "{% set loop_messages = messages[1:] %}"
    "{% else %}"
    "{{ '{system_prompt}' + eos_token }}"
    "{% set loop_messages = messages %}"
    "{% endif %}"
    "{% for message in loop_messages %}"
    "{% if message['role'] == 'user' %}"
    "{{ message['content'] }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ message['content'] + eos_token }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"
    "{% endif %}"
)

# Replace with our specific template:
chat_template = chat_template.replace(
    "'{system_prompt}'", f"'{system_prompt}'"
).replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

[load_model] MODE: TRAIN-SFT -> UNSLOTH_VLLM_STANDBY=0, fast_inference=False
==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Let's see how our chat template behaves on an example:

In [10]:

tokenizer.apply_chat_template(
    [
        {"role": "user", "content": "What is 1+1?"},
        {
            "role": "assistant",
            "content": f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}",
        },
        {"role": "user", "content": "What is 2+2?"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION><|im_end|>What is 1+1?<start_working_out>I think it's 2.<end_working_out><SOLUTION>2</SOLUTION><|im_end|>What is 2+2?<start_working_out>"

### Pre fine-tuning for formatting
We now use a subset of NVIDIA's [Open Math Reasoning dataset](https://huggingface.co/datasets/nvidia/OpenMathReasoning) which was filtered to only include high quality DeepSeek R1 traces.
#
We'll only filter ~59 or so examples to first "prime" / pre fine-tune the model to understand our custom GRPO formatting.

In [11]:
# Put this helper at the top of your script (outside the if block)
from functools import partial

def process_batch(batch, tokenizer, reasoning_start, reasoning_end,
                  solution_start, solution_end, system_prompt):
    messages_list = []
    for i in range(len(batch["problem"])):
        thoughts = batch["generated_solution"][i].replace("<think>", "").replace("</think>", "").strip()
        final_prompt = (reasoning_start + thoughts + reasoning_end
                        + solution_start + batch["expected_answer"][i] + solution_end)
        messages_list.append([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": batch["problem"][i]},
            {"role": "assistant", "content": final_prompt},
        ])
    tokenized_batch = tokenizer.apply_chat_template(messages_list, tokenize=True)
    texts = [tokenizer.decode(ids) for ids in tokenized_batch]
    n_tokens = [len(ids) for ids in tokenized_batch]
    return {"text": texts, "n_tokens": n_tokens}


# Inside your main execution block
if not args.no_sft:
    print("[SFT] Pre‑SFT formatting fine‑tune enabled")

    # [Yêu cầu 4] save config trước khi train
    save_config_and_hyperparams(output_dir, args, tokenizer, extra={"phase": "sft"})

    model, tokenizer = load_model(
        model_path=args.model_path,
        max_seq_length=args.max_seq_length,
        lora_rank=32,
        seed=args.seed,
        mode="train",
        stage="sft" # -> tự tắt standby + fast_inference
    )

    from datasets import load_dataset
    import torch, gc

    # 1. Load & filter numeric answers
    print("Loading dataset...")
    dataset = load_dataset(args.sft_data_path, "default", split="train")
    print(f"Raw size: {len(dataset)}")

    def is_numeric(example):
        try:
            float(example["expected_answer"])
            return True
        except (ValueError, TypeError):
            return False

    dataset = dataset.filter(is_numeric)
    print(f"After numeric filter: {len(dataset)}")


    dataset = dataset.map(
        process_batch,
        batched=True,
        batch_size=1000,
        remove_columns=dataset.column_names,
        num_proc=os.cpu_count(),
        fn_kwargs={
            "tokenizer": tokenizer,
            "reasoning_start": reasoning_start,
            "reasoning_end": reasoning_end,
            "solution_start": solution_start,
            "solution_end": solution_end,
            "system_prompt": system_prompt,
        }
    )
    print(f"After processing: {len(dataset)}")
    dataset = dataset.filter(lambda x: x["n_tokens"] <= args.max_seq_length // 5)
    
    print(f"Data shape: {dataset.shape}")
    print(f"Data examples: {dataset["text"][0]}")
    
    # 4. SFT training (unchanged, but now dataset has "messages" and "text")
    from trl import SFTTrainer, SFTConfig

    print("Starting SFT training...")
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=16,
            gradient_accumulation_steps=1,
            warmup_ratio=0.1,
            num_train_epochs=args.sft_epochs,
            learning_rate=2e-4,
            logging_steps=5,
            optim=args.optim,
            weight_decay=0.001,
            lr_scheduler_type="cosine",
            seed=args.seed,
            report_to="wandb" if args.wandb else "none",
            bf16=True,                             # Enables 16-bit brain float precision (use fp16=True if on older GPUs)
            packing=True,                          # Concatenates short sequences to eliminate padding tokens entirely
            dataloader_num_workers=os.cpu_count() - 1,              # Speeds up data loading by using multiple CPU threads
            dataloader_pin_memory=True,            # Speeds up data transfer from CPU RAM to GPU VRAM
            gradient_checkpointing=True,           # Saves massive VRAM so you can maximize batch sizes (optional, but standard)
            torch_compile=True,
        ),
    )

    # Optional: train only on responses (using unsloth's helper)
    from unsloth.chat_templates import train_on_responses_only
    print(f"Masking to train on response only!")
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

    trainer.train()
    print("SFT training completed.")

    save_dir = "qwen_lora_sft"
    print(f"Save model and tokenizer in folder {save_dir}")
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    !tar -czvf qwen_lora_sft.tar.gz qwen_lora_sft/
else:
    print("[SFT] Skipping pre-SFT (--no-sft): GRPO-from-base comparison arm")

[SFT] Skipping pre-SFT (--no-sft): GRPO-from-base comparison arm


### Data Prep
<a name="Data"></a>
#
We're using Hugging Face's [Open R1 Math dataset](https://huggingface.co/datasets/open-r1/DAPO-Math-17k-Processed). You can also utilize OpenAI's famous [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)

In [12]:

from datasets import load_dataset

dataset = load_dataset(args.grpo_data_path, split="train")
if args.sample_n > 0:
    dataset = dataset.select(range(min(args.sample_n, len(dataset))))
    print(f"[data] Using subset of {len(dataset)} samples")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count'],
    num_rows: 93733
})

Let's look at the first row:

In [13]:
dataset[0]["problem"]

'## Task B-1.3.\n\nA ship traveling along a river has covered $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \\mathrm{~km}$ upstream and $21 \\mathrm{~km}$ downstream, or half an hour more than for traveling $15 \\mathrm{~km}$ upstream and $42 \\mathrm{~km}$ downstream, assuming that both the ship and the river move uniformly.\n\nDetermine the speed of the ship in still water and the speed of the river.'

In [14]:
dataset[0]["solution"]

'## Solution.\n\nLet $t$ be the time required for the boat to travel $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream, $v_{R}$ the speed of the river, and $v_{B}$ the speed of the boat. When the boat is traveling upstream, its speed is $v_{B}-v_{R}$, and when it is traveling downstream, its speed is $v_{B}+v_{R}$.\n\nSince $t=\\frac{s}{v}$, from the given data, we obtain the following system of equations:\n\n$\\left\\{\\begin{array}{l}t=\\frac{24}{v_{B}-v_{R}}+\\frac{28}{v_{B}+v_{R}} \\\\ t+0.5=\\frac{30}{v_{B}-v_{R}}+\\frac{21}{v_{B}+v_{R}} \\\\ t-0.5=\\frac{15}{v_{B}-v_{R}}+\\frac{42}{v_{B}+v_{R}}\\end{array}\\right.$\n\nBy introducing new variables $x=\\frac{3}{v_{B}-v_{R}}, y=\\frac{7}{v_{B}+v_{R}}$, the system transforms into:\n\n$\\left\\{\\begin{array}{l}t=8 x+4 y \\\\ t+0.5=10 x+3 y \\\\ t-0.5=5 x+6 y\\end{array}\\right.$\n\nSubstituting $t$ from the first equation into the remaining two, we get:\n\n$\\left\\{\\begin{array}{l}8 x+4 y+0.5=10 x+3 y \\\\ 8 x+4 y-0.5=5

In GSM8K, we notice all answers like about have a ####, so we extract it. But for the Open R1 dataset, we can skip the below.

In [15]:


def extract_hash_answer(text):
    # if "####" not in text: return None
    # return text.split("####")[1].strip()
    return text


extract_hash_answer(dataset[0]["solution"])

'## Solution.\n\nLet $t$ be the time required for the boat to travel $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream, $v_{R}$ the speed of the river, and $v_{B}$ the speed of the boat. When the boat is traveling upstream, its speed is $v_{B}-v_{R}$, and when it is traveling downstream, its speed is $v_{B}+v_{R}$.\n\nSince $t=\\frac{s}{v}$, from the given data, we obtain the following system of equations:\n\n$\\left\\{\\begin{array}{l}t=\\frac{24}{v_{B}-v_{R}}+\\frac{28}{v_{B}+v_{R}} \\\\ t+0.5=\\frac{30}{v_{B}-v_{R}}+\\frac{21}{v_{B}+v_{R}} \\\\ t-0.5=\\frac{15}{v_{B}-v_{R}}+\\frac{42}{v_{B}+v_{R}}\\end{array}\\right.$\n\nBy introducing new variables $x=\\frac{3}{v_{B}-v_{R}}, y=\\frac{7}{v_{B}+v_{R}}$, the system transforms into:\n\n$\\left\\{\\begin{array}{l}t=8 x+4 y \\\\ t+0.5=10 x+3 y \\\\ t-0.5=5 x+6 y\\end{array}\\right.$\n\nSubstituting $t$ from the first equation into the remaining two, we get:\n\n$\\left\\{\\begin{array}{l}8 x+4 y+0.5=10 x+3 y \\\\ 8 x+4 y-0.5=5

Let's map the dataset! and see the first row:

In [16]:

dataset = dataset.map(
    lambda x: {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": x["problem"]},
        ],
        "answer": extract_hash_answer(x["solution"]),
    }
)
dataset[0]

Map:   0%|          | 0/93733 [00:00<?, ? examples/s]

{'problem': '## Task B-1.3.\n\nA ship traveling along a river has covered $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \\mathrm{~km}$ upstream and $21 \\mathrm{~km}$ downstream, or half an hour more than for traveling $15 \\mathrm{~km}$ upstream and $42 \\mathrm{~km}$ downstream, assuming that both the ship and the river move uniformly.\n\nDetermine the speed of the ship in still water and the speed of the river.',
 'solution': '## Solution.\n\nLet $t$ be the time required for the boat to travel $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream, $v_{R}$ the speed of the river, and $v_{B}$ the speed of the boat. When the boat is traveling upstream, its speed is $v_{B}-v_{R}$, and when it is traveling downstream, its speed is $v_{B}+v_{R}$.\n\nSince $t=\\frac{s}{v}$, from the given data, we obtain the following system of equations:\n\n$\\left\\{\\begin{array}{l}t=\\frac{24}{v_{B}-v_{R}}+\\fra

We create a regex format to match the reasoning sections and answers:

In [17]:

import re

# Add optional EOS token matching
solution_end_regex = (
    r"</SOLUTION>[\s]{0,}" + "(?:" + re.escape(tokenizer.eos_token) + ")?"
)

match_format = re.compile(
    rf"{reasoning_end}.*?"
    rf"{solution_start}(.+?){solution_end_regex}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL,
)
match_format

re.compile(r'<end_working_out>.*?<SOLUTION>(.+?)</SOLUTION>[\s]{0,}(?:<\|im_end\|>)?[\s]{0,}$',
re.MULTILINE|re.DOTALL|re.UNICODE)

We verify it works:

In [18]:

match_format.findall(
    f"Let me think!<end_working_out><SOLUTION>\n2\n</SOLUTION>",
)

match_format.findall(
    f"<start_working_out>Let me think!<end_working_out><SOLUTION>  2  </SOLUTION>\n\n",
)

['  2  ']

We now want to create a reward function to match the format exactly - we reward it with 3 points if it succeeds:

In [19]:


def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None:
            score += 3.0
        scores.append(score)
    return scores

If it fails, we want to reward the model if it at least follows the format partially, by counting each symbol:

In [20]:


def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!

        # No need to reward the opening tag since we always prepend it!
        # score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end) == 1 else -1.0
        score += 0.5 if response.count(solution_start) == 1 else -1.0
        score += 0.5 if response.count(solution_end) == 1 else -1.0
        scores.append(score)
    return scores

Finally, we want to extract the generated answer, and reward or penalize it! We also reward it based on how close the answer is to the true one via ratios:

In [21]:


def check_answer(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1) if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(-2.0)
            continue
        # Correct answer gets 5 points!
        if guess == true_answer:
            score += 5.0
        # Match if spaces are seen, but less reward
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # We also reward it if the answer is close via ratios!
            # Ie if the answer is within some range, reward it!
            try:
                ratio = float(guess) / float(true_answer)
                if ratio >= 0.9 and ratio <= 1.1:
                    score += 2.0
                elif ratio >= 0.8 and ratio <= 1.2:
                    score += 1.5
                else:
                    score -= 2.5  # Penalize wrong answers
            except:
                score -= 4.5  # Penalize
        scores.append(score)
    return scores

Also sometimes it might not be 1 number as the answer, but like a sentence for example "The solution is $20" -> we extract 20.
#
We also remove possible commas for example as in 123,456

In [22]:

match_numbers = re.compile(
    solution_start + r".*?[\s]{0,}([-]?[\d\.\,]{1,})", flags=re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  -0.234  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>17</SOLUTION>"))

['0.34']
['123,456']
['-0.234']
['17']


We now prepare our main function which will print out the generated responses and the true answer, along with another reward function which converts text to float via `float` and sees if it's the same.

In [23]:

global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5


def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1) if (guess := match_numbers.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    # Print only every few steps
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            "*" * 20 + f"Question:\n{question}",
            f"\nAnswer:\n{answer[0]}",
            f"\nResponse:\n{responses[0]}",
            f"\nExtracted:\n{extracted_responses[0]}",
        )
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        # Convert to numbers
        try:
            true_answer = float(true_answer.strip())
            # Remove commas like in 123,456
            guess = float(guess.strip().replace(",", ""))
            scores.append(3.5 if guess == true_answer else -1.5)
        except:
            scores.append(0)
            continue
    return scores

Get the top 90% prompt length so we don't accidentally truncate them!
#
Ie we'll remove the top 10% long prompts.

In [24]:

tokenized = dataset.map(
    lambda x: {
        "tokens": tokenizer.apply_chat_template(
            x["prompt"], add_generation_prompt=True, tokenize=True
        )
    },
    batched=True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L": len(x["tokens"])})

import numpy as np

maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

Map:   0%|          | 0/93733 [00:00<?, ? examples/s]

You are given a problem.
Think about the problem and provide your working out.
Place it between <start_working_out> and <end_working_out>.
Then, provide your solution between <SOLUTION></SOLUTION><|im_end|>## Task B-1.3.

A ship traveling along a river has covered $24 \mathrm{~km}$ upstream and $28 \mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \mathrm{~km}$ upstream and $21 \mathrm{~km}$ downstream, or half an hour more than for traveling $15 \mathrm{~km}$ upstream and $42 \mathrm{~km}$ downstream, assuming that both the ship and the river move uniformly.

Determine the speed of the ship in still water and the speed of the river.<start_working_out>


Map:   0%|          | 0/93733 [00:00<?, ? examples/s]

Max Length =  205


### Train the model
Now set up GRPO Trainer and all configurations!

In [25]:

max_prompt_length = maximum_length + 1  # + 1 just in case!
max_completion_length = args.max_seq_length - max_prompt_length

from vllm import SamplingParams

vllm_sampling_params = SamplingParams(
    min_p=0.1,
    top_p=1.0,
    top_k=-1,
    seed=args.seed,
    stop=[tokenizer.eos_token],
    include_stop_str_in_output=True,
)

from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=args.temperature,
    learning_rate=args.lr,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim=args.optim,
    logging_steps=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Increase to 4 for smoother training
    num_generations=args.num_generations,  # Ablation: group size
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps=args.max_steps,
    save_steps=args.max_steps,
    report_to="wandb" if args.wandb else "none",  # Weights & Biases
    output_dir=str(output_dir),
    seed=args.seed,
    run_name=RUN_NAME,
    # === Ablation knobs (research-backed) ===
    beta=args.beta,  # KL divergence penalty (0 => KL not computed/logged)
    loss_type=args.loss_type,  # grpo | bnpo | dr_grpo | dapo (normalization variants)
    importance_sampling_level=args.importance_sampling_level,  # token vs sequence advantage credit
    scale_rewards="group",  # group-normalized advantages (DeepSeekMath scheme)
    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

Unsloth: The DAPO paper recommends `mask_truncated_completions = True` - we will set it.
Unsloth: The DAPO paper recommends `epsilon_high = 0.28` - we will set it.


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!
#
You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!
#
| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [26]:

# === WandB login (API key read from environment automatically) ===
if args.wandb:
    try:
        import wandb

        if os.environ.get("WANDB_API_KEY"):
            wandb.login(key=os.environ["WANDB_API_KEY"])
        print(
            f"[wandb] Logging to project={os.environ.get('WANDB_PROJECT')} "
            f"group={os.environ.get('WANDB_RUN_GROUP')} name={RUN_NAME} mode={os.environ.get('WANDB_MODE', 'online')}"
        )
    except Exception as e:
        print(f"[wandb] Init failed ({e}) — continuing without wandb")

# === Custom callback: dump EVERY logged metric per run ===
from transformers import TrainerCallback


class MetricsDumpCallback(TrainerCallback):
    """Serializes all trainer metrics to outputs/<run_name>/metrics.json.

    GRPOTrainer already logs (no duplication here): reward, reward_std,
    kl (only when beta != 0), entropy, completions/*, rewards/{func}/*,
    clip_ratio/*, loss, grad_norm, learning_rate, epoch.
    """

    def __init__(self, run_name, args_namespace, out_dir):
        self.run_name = run_name
        self.config = vars(args_namespace)
        self.out_dir = pathlib.Path(out_dir)
        self.metrics_history = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        entry = {"step": state.global_step}
        entry.update({k: v for k, v in logs.items() if isinstance(v, (int, float))})
        self.metrics_history.append(entry)

    def on_train_end(self, args, state, control, **kwargs):
        dump = {
            "run_name": self.run_name,
            "config": self.config,
            "final_metrics": self.metrics_history[-1] if self.metrics_history else {},
            "metrics_history": self.metrics_history,
        }
        out_path = self.out_dir / "metrics.json"
        with open(out_path, "w") as f:
            json.dump(dump, f, indent=2, default=str)
        print(f"[metrics] Saved to {out_path}")


metrics_callback = MetricsDumpCallback(RUN_NAME, args, output_dir)

# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

save_config_and_hyperparams(output_dir, args, tokenizer, extra={"phase": "grpo"})

model, tokenizer = load_model(
    model_path=args.model_path,
    max_seq_length=args.max_seq_length,
    lora_rank=32,
    seed=args.seed,
    mode="train",
    stage="grpo" # -> tự bật standby + fast_inference
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args=training_args,
    train_dataset=dataset,
    callbacks=[metrics_callback],
    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)
trainer.train()

# === Post-training metrics fallback (covers callback non-firing edge cases) ===
if not (output_dir / "metrics.json").exists():
    fallback_metrics = {
        "run_name": RUN_NAME,
        "config": vars(args),
        "final_metrics": trainer.state.log_history[-1]
        if trainer.state.log_history
        else {},
        "metrics_history": trainer.state.log_history,
    }
    with open(output_dir / "metrics.json", "w") as f:
        json.dump(fallback_metrics, f, indent=2, default=str)
    print(f"[metrics] Fallback dump saved to {output_dir / 'metrics.json'}")

if args.wandb:
    try:
        import wandb

        wandb.finish()
    except Exception:
        pass

[tracking] Saved config to outputs/beta0.0_ng8_loss_dapo_is_token_temp1.0_lr1e-05_optadamw_torch_noSFT/config.json
[tracking] Saved hyperparams to outputs/beta0.0_ng8_loss_dapo_is_token_temp1.0_lr1e-05_optadamw_torch_noSFT/hyperparams.json
[load_model] MODE: TRAIN-GRPO -> UNSLOTH_VLLM_STANDBY=1, fast_inference=True
INFO 08-21 17:57:44 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading /kaggle/input/models/qwen-lm/qwen-3/transformers/1.7b-base/1 with actual GPU uti

[W821 17:57:56.317390423 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


INFO 08-21 17:57:57 [topk_topp_sampler.py:51] Using FlashInfer for top-p & top-k sampling.
INFO 08-21 17:57:57 [gpu_model_runner.py:4481] Starting to load model /kaggle/input/models/qwen-lm/qwen-3/transformers/1.7b-base/1...
INFO 08-21 17:57:58 [cuda.py:317] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 08-21 17:57:58 [flash_attn.py:598] Using FlashAttention version 2


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


...

Unsloth: Just some info: will skip parsing ['post_layernorm', 'attention_norm', 'norm', 'q_norm', 'post_attention_layernorm', 'input_layernorm', 'ffn_norm', 'pre_feedforward_layernorm', 'layer_norm2', 'k_norm', 'post_feedforward_layernorm', 'norm2', 'norm1', 'layer_norm1']
Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['post_layernorm', 'attention_norm', 'norm', 'cross_attn_input_layernorm', 'q_norm', 'post_attention_layernorm', 'input_layernorm', 'ffn_norm', 'pre_feedforward_layernorm', 'layer_norm2', 'cross_attn_post_attention_layernorm', 'k_norm', 'post_feedforward_layernorm', 'norm2', 'norm1', 'layer_norm1']


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 84,442 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 34,865,152 of 1,755,440,128 (1.99% trained)


WARNING 08-21 17:59:53 [input_processor.py:141] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!
********************Question:
When $x=\frac{1}{5}$, the value of the expression $\frac{x^{2}-4}{x^{2}-2 x}$ is
(A) 0.4
(B) -0.52
(C) -5
(D) 10
(E) 11 
Answer:
Solution 1

We first simplify the expression:

$$
\frac{x^{2}-4}{x^{2}-2 x}=\frac{(x+2)(x-2)}{x(x-2)}=\frac{x+2}{x}=\frac{x}{x}+\frac{2}{x}=1+\frac{2}{x}
$$

(We can cancel the factor of $x-2$ since $x$ is not equal to 2.)

Substituting $x=\frac{1}{5}$, we get $1+\frac{2}{\left(\frac{1}{5}\right)}=1+10=11$.

## Solution 2

Substituting $x=\frac{1}{5}$,

$$
\frac{x^{2}-4}{x^{2}-2 x}=\frac{\frac{1}{25}-4}{\frac{1}{25}-\frac{2}{5}}=\frac{\frac{1}{25}-\frac{100}{25}}{\frac{1}{25}-\frac{

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_exactly / mean,rewards / match_format_exactly / std,rewards / match_format_approximately / mean,rewards / match_format_approximately / std,rewards / check_answer / mean,rewards / check_answer / std,rewards / check_numbers / mean,rewards / check_numbers / std
1,-0.640900,-6.687500,1.514719,553.375000,6.000000,3175.000000,0.000000,553.375000,6.000000,3175.000000,0.000000,0.000000,0.000000,-2.343750,1.338142,-2.000000,0.000000,-2.343750,0.625000
2,-0.092400,-6.062500,2.741197,1050.312500,46.000000,2598.000000,0.000000,1050.312500,46.000000,2598.000000,0.000000,0.187500,0.750000,-2.062500,1.631717,-2.156250,0.625000,-2.031250,1.007782
3,-0.129800,-6.343750,1.910045,1056.187500,44.000000,4579.000000,0.000000,1056.187500,44.000000,4579.000000,0.000000,0.000000,0.000000,-2.156250,1.220912,-2.000000,0.000000,-2.187500,0.853913
4,0.518100,-5.906250,2.244034,2344.000000,9.000000,16178.000000,0.062500,1421.733398,9.000000,8353.000000,0.000000,0.000000,0.000000,-1.875000,1.596872,-2.000000,0.000000,-2.031250,1.007782
5,0.050500,-6.093750,1.680152,1099.625000,34.000000,8399.000000,0.000000,1099.625000,34.000000,8399.000000,0.000000,0.187500,0.750000,-2.250000,0.948683,-2.156250,0.625000,-1.875000,1.118034
6,-0.089700,-6.500000,2.012096,1136.062500,18.000000,3458.000000,0.000000,1136.062500,18.000000,3458.000000,0.000000,0.187500,0.750000,-2.343750,1.091158,-2.156250,0.625000,-2.187500,0.853913
7,0.055600,-7.406250,0.265165,864.875000,7.000000,3088.000000,0.000000,864.875000,7.000000,3088.000000,0.000000,0.000000,0.000000,-2.906250,0.375000,-2.000000,0.000000,-2.500000,0.000000
8,-0.026500,-5.593750,1.948382,831.437500,6.000000,1570.000000,0.000000,831.437500,6.000000,1570.000000,0.000000,0.000000,0.000000,-1.875000,1.161895,-2.000000,0.000000,-1.718750,1.196784
9,0.253600,-6.375000,2.179262,623.625000,17.000000,1974.000000,0.000000,623.625000,17.000000,1974.000000,0.000000,0.000000,0.000000,-2.343750,1.338142,-2.000000,0.000000,-2.031250,1.007782
10,-0.014000,-5.687500,2.445644,664.875000,199.000000,1621.000000,0.000000,664.875000,199.000000,1621.000000,0.000000,0.375000,1.024695,-1.875000,1.500000,-2.312500,0.853913,-1.875000,1.118034


********************Question:
3. Solve the equation:

$$
\frac{1}{\sqrt{5^{x}+12^{x}}}+\sqrt{x+7}=\frac{14}{13}+\sqrt{x+2}
$$ 
Answer:
3. $\frac{1}{\sqrt{5^{x}+12^{x}}}+\sqrt{x+7}-\sqrt{x+2}=\frac{14}{13}$ or $\frac{1}{\sqrt{5^{x}+12^{x}}}+\frac{5}{\sqrt{x+7}+\sqrt{x+2}}=\frac{14}{13}$ $.2 p$

$x=2$ verifies the equation, and the function on the left side is strictly decreasing, so $x=2$ is the only solution

$5 p$ 
Response:
Here is how I would solve it:

// The given equation has two square root terms, so I would start by squaring both sides to eliminate them.

// First, I would isolate one square root and square both sides.

// Expanding out the left side, I get:

$$\frac{1}{5^x + 12^x} + 52 + 16 \cdot 2^{2x} = \left(\frac{14}{13} + \sqrt{x+2}\right)^2$$

// Next, I would simplify the equation by expanding out the right side.

// Grouping terms and simplifying, I get:

$$\frac{1190 + 2 \cdot 52 \cdot 2^{2x}}{169 - 84 \sqrt{x+2} + 12x (x+2)} = \frac{144}{169} + \frac{4 \cdot \sqrt{x+

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [27]:

text = "What is the sqrt of 101?"

from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=1024,
)
output = (
    model.fast_generate(
        [text],
        sampling_params=sampling_params,
        lora_request=None,
    )[0]
    .outputs[0]
    .text
)

output

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

' - Find 225 Answers & Solutions | LearnPick Resources\nToday Chat\nSend Your Question. Get a quick response.'

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [28]:

model.save_lora(str(output_dir / "grpo_saved_lora"))

Verify LoRA is actually trained!

In [29]:

from safetensors import safe_open

tensors = {}
with safe_open(
    str(output_dir / "grpo_saved_lora" / "adapter_model.safetensors"), framework="pt"
) as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert n_zeros.item() != tensor.numel()

Now we load the LoRA and test:

In [30]:

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,  # Must add for generation
    tokenize=False,
)
from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=2048,
)
output = (
    model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=model.load_lora(str(output_dir / "grpo_saved_lora")),
    )[0]
    .outputs[0]
    .text
)

output

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'<start_working_out>\nThe square root of 101 is an irrational number, which means it cannot be expressed as a simple fraction and has non-repeating, non-terminating decimal expansion. Using a calculator or a more precise method, we find that:\n\n\\[\n\\sqrt{101} \\approx 10.049875621120888\n\\]\n\nThis value is rounded to 15 decimal places for practical purposes.\n<end_working_out>\n\n<SOLUTION>\nThe square root of 101 is approximately \\(\\sqrt{101} \\approx 10.049875621120888\\).\n</SOLUTION>'

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!
#
<a name="Save"></a>
### Saving to float16 for VLLM
#
We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [31]:

# Merge to 16bit
if False:
    model.save_pretrained_merged(
        "qwen_finetune_16bit",
        tokenizer,
        save_method="merged_16bit",
    )
if False:
    model.push_to_hub_merged(
        "HF_USERNAME/qwen_finetune_16bit",
        tokenizer,
        save_method="merged_16bit",
        token="YOUR_HF_TOKEN",
    )

# Merge to 4bit
if False:
    model.save_pretrained_merged(
        "qwen_finetune_4bit",
        tokenizer,
        save_method="merged_4bit",
    )
if False:
    model.push_to_hub_merged(
        "HF_USERNAME/qwen_finetune_4bit",
        tokenizer,
        save_method="merged_4bit",
        token="YOUR_HF_TOKEN",
    )

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False:
    model.push_to_hub("HF_USERNAME/qwen_lora", token="YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token="YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.
#
Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.
#
[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [32]:

# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf(
        "qwen_finetune",
        tokenizer,
    )
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", tokenizer, token="YOUR_HF_TOKEN"
    )

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method="f16")
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune",
        tokenizer,
        quantization_method="f16",
        token="YOUR_HF_TOKEN",
    )

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method="q4_k_m")
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune",
        tokenizer,
        quantization_method="q4_k_m",
        token="YOUR_HF_TOKEN",
    )

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune",  # Change hf to your username!
        tokenizer,
        quantization_method=[
            "q4_k_m",
            "q8_0",
            "q5_k_m",
        ],
        token="YOUR_HF_TOKEN",
    )

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.
#
And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!
#
Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!
#
<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>
#
  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
#
  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).